In [3]:
import torch
print(torch.cuda.is_available())

False


In [2]:
import pandas as pd
import torch

### **Dealing with the dataset**

In [5]:
df = pd.read_csv('data/fra.txt',sep = '\t')

In [6]:
df.head()

,Go.,Va !,CC-BY 2.0 (France) Attribution: tatoeba.org #2877272 (CM) & #1158250 (Wittydev)
0,Go.,Marche.,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
1,Go.,En route !,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
2,Go.,Bouge !,CC-BY 2.0 (France) Attribution: tatoeba.org #2...
3,Hi.,Salut !,CC-BY 2.0 (France) Attribution: tatoeba.org #5...
4,Hi.,Salut.,CC-BY 2.0 (France) Attribution: tatoeba.org #5...


In [7]:
df = df.iloc[:,:-1]

In [8]:
df.sample(10)

,Go.,Va !
167797,We left home early in the morning.,Nous quittâmes la maison de bon matin.
213595,I won't be spending Christmas with my family.,Je ne passerai pas Noël avec ma famille.
14261,You seem smart.,Vous avez l'air intelligente.
77685,"In the end, Tom gave up.",Tom a fini par céder.
182278,I didn't know that you were so tired.,Je ne savais pas que tu étais si fatiguée.
227177,I got a stomach tumor and had to have it opera...,J'ai eu une tumeur de l'estomac et j'ai dû la ...
35879,It was really dark.,C'était vraiment sombre.
50989,I'll keep it with me.,Je vais le garder avec moi.
180409,We're not lost. I know where we are.,Nous ne nous sommes pas perdus. Je sais où nou...
176251,He was in favor of equality for all.,Il était en faveur de l'égalité pour tous.


In [9]:
df.to_csv('data/Eng-Fra.csv',index = False)

In [10]:
df = pd.read_csv('data/Eng-Fra.csv')
df.head()

,English,French
0,Go.,Va !
1,Go.,Marche.
2,Go.,En route !
3,Go.,Bouge !
4,Hi.,Salut !


In [11]:
df.columns

Index(['English', 'French'], dtype='object')

In [12]:
df.shape

(239189, 2)

### **Preprocessing**

In [13]:
from preprocessing.normalize import normalize_text

df['English'] = df['English'].apply(normalize_text)
df['French'] = df['French'].apply(normalize_text)

In [14]:
df.head()

,English,French
0,go,va
1,go,marche
2,go,en route
3,go,bouge
4,hi,salut


### **Vocabulary Building**

In [15]:
from preprocessing.vocabulary import Vocabulary

eng_vocab = Vocabulary('english')
fr_vocab = Vocabulary('french')

In [18]:
for eng, fr in zip(df['English'], df['French']):
    eng_vocab.add_sentence(eng)
    fr_vocab.add_sentence(fr)

In [19]:
print("Engish Vocabulary Size: ",eng_vocab.vocab_size)
print("French Vocabulary Size: ",fr_vocab.vocab_size)

Engish Vocabulary Size:  4
French Vocabulary Size:  4


In [8]:
from preprocessing.sentence_to_tensor import sentence_to_tensor
print("English")
print("Input: i am cold")
tensor_eng = sentence_to_tensor(eng_vocab,'i am cold')
print("Tensor: ",tensor_eng)
print("Tensor Shape: ",tensor_eng.shape)
print("Tensor Datatype: ",tensor_eng.dtype)
print('\n------------------\n')
print("French")
print("Input: je suis froid")
tensor_fr = sentence_to_tensor(fr_vocab,'je suis froid')
print("Tensor: ",tensor_fr)
print("Tensor Shape: ",tensor_fr.shape)
print("Tensor Datatype: ",tensor_fr.dtype)

English
Input: i am cold
Tensor:  tensor([ 19, 150, 207,   1])
Tensor Shape:  torch.Size([4])
Tensor Datatype:  torch.int64

------------------

French
Input: je suis froid
Tensor:  tensor([ 46, 106, 545,   1])
Tensor Shape:  torch.Size([4])
Tensor Datatype:  torch.int64


In [8]:
# verifying word mapping
print("English Vocabulary")
print(eng_vocab.index2word[19])
print(eng_vocab.index2word[150])
print(eng_vocab.index2word[207])
print(eng_vocab.index2word[1])
print('---------------------',end = '\n')
print("French Vocabulary")
print(fr_vocab.index2word[46])
print(fr_vocab.index2word[106])
print(fr_vocab.index2word[545])
print(fr_vocab.index2word[1])

English Vocabulary
i
am
cold
EOS
---------------------
French Vocabulary
je
suis
froid
EOS


### **The Encoder Part**

In [9]:
from models.encoder import Encoder

encoder = Encoder(vocab_size = eng_vocab.vocab_size, embedding_dim = 256, hidden_size = 512)
input_tensor = sentence_to_tensor(eng_vocab,'i am cold')
input_tensor = input_tensor.unsqueeze(0) # adds a batch dimension

encoder_outputs, encoder_hidden = encoder(input_tensor)

In [10]:
print("Input shape:", input_tensor.shape)
print("Encoder outputs shape:", encoder_outputs.shape)
print("Hidden shape:", encoder_hidden.shape)

Input shape: torch.Size([1, 4])
Encoder outputs shape: torch.Size([1, 4, 512])
Hidden shape: torch.Size([1, 1, 512])


In [11]:
print(encoder_outputs)

tensor([[[-0.1357, -0.1254, -0.3636,  ...,  0.4629,  0.0756,  0.1026],
         [-0.3123, -0.2231, -0.3111,  ...,  0.5390,  0.1992,  0.2579],
         [-0.5360,  0.3281, -0.2878,  ...,  0.2897,  0.1088, -0.2709],
         [-0.3123,  0.1545, -0.1856,  ...,  0.2229,  0.0497, -0.0057]]],
       grad_fn=<TransposeBackward1>)


In [13]:
print(encoder_hidden)

tensor([[[-3.1300e-01, -4.0522e-01,  1.7142e-02, -4.9370e-01, -2.4274e-01,
          -9.2513e-02,  1.5109e-01,  1.8159e-01, -2.0438e-01, -1.3095e-01,
          -6.0902e-02, -2.6568e-01,  2.2976e-01, -3.4767e-01,  3.9308e-01,
           3.5519e-01,  3.3935e-01,  6.5468e-02,  3.2292e-01, -4.0314e-01,
          -2.4842e-01,  2.9810e-01,  5.0345e-02,  2.0033e-01,  2.9292e-01,
           3.1641e-01,  9.5965e-04,  1.5583e-01,  2.0398e-01,  1.8466e-01,
          -1.8714e-01,  3.1300e-02, -3.2667e-01,  1.4779e-02,  1.3582e-02,
           5.2242e-02,  9.4495e-02,  3.4118e-01,  1.2148e-01, -6.6933e-02,
          -2.8573e-01,  6.7898e-02,  4.9664e-02,  6.9269e-02,  2.1963e-01,
          -8.3857e-02, -1.5241e-01,  2.2065e-02,  3.7159e-01,  7.7296e-02,
          -1.2733e-01, -3.1591e-01,  2.4678e-01, -5.4763e-01,  4.3765e-01,
          -2.4264e-02,  1.6785e-01, -2.0082e-01,  8.0485e-02, -6.2078e-02,
           1.3713e-01, -8.4408e-02,  9.6767e-02,  1.5436e-01,  1.3192e-01,
           2.0021e-01, -2

### **The Decoder Part**

In [11]:
from models.decoder import Decoder

decoder = Decoder(vocab_size = fr_vocab.vocab_size, embedding_dim = 256, hidden_size = 512)
decoder_input = torch.tensor([[fr_vocab.word2index['SOS']]])
decoder_output, decoder_hidden = decoder(decoder_input, encoder_hidden)

In [12]:
print("Decoder output shape:", decoder_output.shape)
print("Decoder hidden shape:", decoder_hidden.shape)

Decoder output shape: torch.Size([1, 1, 35321])
Decoder hidden shape: torch.Size([1, 1, 512])


In [13]:
predicted_token = decoder_output.argmax(-1)
print(predicted_token)

tensor([[33061]])


In [14]:
print(fr_vocab.index2word[predicted_token.item()])

universités


### **The Seq2Seq Model**

In [15]:
from models.seq2seq import Seq2Seq
device = torch.device("cpu")

encoder = Encoder(vocab_size = eng_vocab.vocab_size, embedding_dim=256, hidden_size = 512)
decoder = Decoder(vocab_size = fr_vocab.vocab_size, embedding_dim= 256, hidden_size = 512)

model = Seq2Seq(encoder, decoder, device)

In [32]:
input_tensor = sentence_to_tensor(eng_vocab,'i am cold')
input_tensor = input_tensor.unsqueeze(0) 

target_tensor = sentence_to_tensor(fr_vocab,'je suis froid')
target_tensor = target_tensor.unsqueeze(0) 

src = input_tensor
trg = target_tensor

output = model(src,trg)

In [20]:
print(output.shape)
print(trg.shape)

torch.Size([1, 4, 35321])
torch.Size([1, 4])


In [17]:
output = output.view(-1, output.shape[-1])
target = trg.view(-1)

In [18]:
print(trg)
print(trg.shape)

tensor([[ 46, 106, 545,   1]])
torch.Size([1, 4])


In [13]:
print(output.shape)
print(target.shape)

torch.Size([4, 35321])
torch.Size([4])


In [18]:
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [19]:
# backpropagation
for i in range(0,25):
    optimizer.zero_grad()
    output = model(src,trg)
    output = output.view(-1, output.shape[-1])
    target = trg.view(-1)
    loss = criterion(output, target)
    print(loss.item())
    loss.backward()
    optimizer.step()
    pred = output.argmax(1)
    for token in pred:
        print(fr_vocab.index2word[token.item()])

10.396927833557129
SOS
cesserait
railleries
provisions
10.29338264465332
SOS
suis
écrismoi
refusé
9.836687088012695
SOS
suis
froid
EOS
9.314943313598633
SOS
suis
froid
EOS
8.756999969482422
SOS
suis
froid
EOS
8.17231559753418
SOS
suis
froid
EOS
7.5656418800354
SOS
suis
froid
EOS
6.940788745880127
SOS
suis
froid
EOS
6.301896095275879
SOS
suis
froid
EOS
5.654910564422607
SOS
suis
froid
EOS
5.0106611251831055
SOS
suis
froid
EOS
4.389815330505371
SOS
suis
froid
EOS
3.8257646560668945
SOS
suis
froid
EOS
3.3577637672424316
SOS
suis
froid
EOS
3.017045021057129
SOS
suis
froid
EOS
2.809523105621338
SOS
suis
froid
EOS
2.704559326171875
SOS
suis
froid
EOS
2.657414197921753
SOS
suis
froid
EOS
2.63683819770813
SOS
suis
froid
EOS
2.6275978088378906
SOS
suis
froid
EOS
2.6232213973999023
SOS
suis
froid
EOS
2.6210618019104004
SOS
suis
froid
EOS
2.619882106781006
SOS
suis
froid
EOS
2.619107246398926
SOS
suis
froid
EOS
2.618734121322632
SOS
suis
froid
EOS


### **Creating Dataset pairs**

In [20]:
pairs = []
for eng, fr in zip(df['English'][:5000], df['French'][:5000]):
    pairs.append((eng,fr))

In [29]:
print((pairs))

[('go', 'va'), ('go', 'marche'), ('go', 'en route'), ('go', 'bouge'), ('hi', 'salut'), ('hi', 'salut'), ('run', 'cours'), ('run', 'courez'), ('run', 'prenez vos jambes à vos cous'), ('run', 'file'), ('run', 'filez'), ('run', 'cours'), ('run', 'fuyez'), ('run', 'fuyons'), ('run', 'cours'), ('run', 'courez'), ('run', 'prenez vos jambes à vos cous'), ('run', 'file'), ('run', 'filez'), ('run', 'cours'), ('run', 'fuyez'), ('run', 'fuyons'), ('who', 'qui'), ('wow', 'ça alors'), ('wow', 'waouh'), ('wow', 'wah'), ('duck', 'à terre'), ('duck', 'baissetoi'), ('duck', 'baissezvous'), ('fire', 'au feu'), ('help', 'à laide'), ('hide', 'cachetoi'), ('hide', 'cachezvous'), ('jump', 'saute'), ('jump', 'saute'), ('stop', 'ça suffit'), ('stop', 'stop'), ('stop', 'arrêtetoi'), ('wait', 'attends'), ('wait', 'attendez'), ('wait', 'attendez'), ('wait', 'attends'), ('wait', 'attendez'), ('wait', 'attends'), ('wait', 'attendez'), ('begin', 'commencez'), ('begin', 'commence'), ('go on', 'poursuis'), ('go on', 

In [21]:
from preprocessing.dataset import TranslationDataset
dataset = TranslationDataset(pairs, eng_vocab, fr_vocab)

src, trg = dataset[0]

print(src)
print(trg)

tensor([4, 1])
tensor([4, 1])


### **Creating DataLoader**

In [22]:
from torch.utils.data import DataLoader
from preprocessing.dataset import TranslationDataset, collate_fn

In [ ]:
dataset = TranslationDataset(pairs, eng_vocab, fr_vocab)

dataloader = DataLoader(
    dataset,
    batch_size = 32,
    shuffle = True,
    collate_fn = collate_fn
)

In [24]:
for src, trg in dataloader:
    print(src.shape)
    print(trg.shape)
    break

torch.Size([32, 4])
torch.Size([32, 5])


### **Full Training Loop**

In [25]:
PAD_IDX = 2 # the tokens which should be ignored and must not affect the training process

criterion = torch.nn.CrossEntropyLoss(ignore_index = PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [27]:
from training.train import train

train(model, num_epochs = 10, dataloader = dataloader, optimizer = optimizer, criterion = criterion)

Epoch: 1, Loss: 6.5283
Epoch: 2, Loss: 5.5478
Epoch: 3, Loss: 5.3128
Epoch: 4, Loss: 5.1442
Epoch: 5, Loss: 5.0595
Epoch: 6, Loss: 4.9781
Epoch: 7, Loss: 4.9649
Epoch: 8, Loss: 4.9067
Epoch: 9, Loss: 4.8927
Epoch: 10, Loss: 4.8558


### **TranslateSentence Function**

In [ ]:
def translate_sentence(sentence):

    src = sentence_to_tensor(eng_vocab, sentence)
    src = (src.unsqueeze(0)).to(device)

    encoder_outputs, hidden = encoder(src)

    decoder_input =( torch.tensor([[fr_vocab.word2index["SOS"]]]).to(device))

    translated_tokens = []

    for _ in range(20):

        output, hidden, attn_weights = decoder(decoder_input, hidden, encoder_outputs)

        pred_token = output.argmax(2)

        pred_idx = pred_token.item()

        if pred_idx == fr_vocab.word2index["EOS"]:
            break

        translated_tokens.append(fr_vocab.index2word[pred_idx])

        decoder_input = pred_token

    return " ".join(translated_tokens)

In [42]:
translate_sentence("hii there")

''